In [1]:

# OUTPUT DASHBOARD

import pandas as pd

# Load our complete rankings
df = pd.read_csv("data/outputs/full_rankings_with_fit.csv")

print(f"Total players: {len(df)}")
print(f"Tier 1: {len(df[df['tier'] == 'Tier 1'])}")
print(f"Tier 2: {len(df[df['tier'] == 'Tier 2'])}")
print(f"Tier 3: {len(df[df['tier'] == 'Tier 3 - Development'])}")



Total players: 220
Tier 1: 149
Tier 2: 25
Tier 3: 46


In [2]:

positions = ['C', 'SF', 'PF', 'PG', 'SG']

for pos in positions:
    t1 = df[(df['tier'] == 'Tier 1') & 
            (df['Position'] == pos)].copy()
    t1 = t1.dropna(subset=['fit_score'])
    t1 = t1.sort_values('fit_score', ascending=False).head(10)
    
    t2 = df[(df['tier'] == 'Tier 2') & 
            (df['Position'] == pos)].copy()
    t2 = t2.sort_values('composite_score', ascending=False)

    print(f"\n{'='*65}")
    print(f"  {pos} RECRUITING BOARD")
    print(f"{'='*65}")
    print(f"  Tier 1 players found: {len(t1)}")
    print(f"  Tier 2 players found: {len(t2)}")

    if len(t1) > 0:
        print(f"\n  TIER 1 — Full Metrics")
        print(f"  {'Name':<22} {'Team':<22} {'Fit':<8} {'Def':<7} {'Off'}")
        print(f"  {'-'*60}")
        for i, (_, row) in enumerate(t1.iterrows(), 1):
            print(f"  {i}. {row['Name']:<20} {str(row['Team']):<22} {row['fit_score']:<8.1f} {row['defense_score']:<7.1f} {row['offense_score']:.1f}")

    if len(t2) > 0:
        print(f"\n  TIER 2 — ON3 Rating Only")
        print(f"  {'Name':<22} {'Year':<8} {'ON3 Rating'}")
        print(f"  {'-'*40}")
        for _, row in t2.iterrows():
            print(f"  {row['Name']:<22} {row['Year']:<8} {row['ON3 Rating']:.0f}")



  C RECRUITING BOARD
  Tier 1 players found: 4
  Tier 2 players found: 1

  TIER 1 — Full Metrics
  Name                   Team                   Fit      Def     Off
  ------------------------------------------------------------
  1. Ernest Udeh Jr.      Miami (Fla.)           81.5     89.2    58.9
  2. Micah Handlogten     Florida                77.2     74.3    63.0
  3. Melian Martinez      Louisiana Tech         56.6     59.4    28.3
  4. Jaden Smith          Saint Joseph's         43.0     36.1    18.2

  TIER 2 — ON3 Rating Only
  Name                   Year     ON3 Rating
  ----------------------------------------
  Ja'Quay Randolph       JR       89

  SF RECRUITING BOARD
  Tier 1 players found: 10
  Tier 2 players found: 2

  TIER 1 — Full Metrics
  Name                   Team                   Fit      Def     Off
  ------------------------------------------------------------
  1. RJ Godfrey           Clemson                81.7     88.0    73.8
  2. Tyler Nickel         Va

In [3]:
# TIER 3 DEVELOPMENT PROSPECTS (C AND PF ONLY) 

print("=" * 65)
print("  DEVELOPMENT PROSPECTS — C AND PF")
print("  Ranked by: Youth first, then Height")
print("=" * 65)

def height_to_inches(h):
    try:
        parts = str(h).split('-')
        return int(parts[0]) * 12 + int(parts[1])
    except:
        return 0

for pos in ['C', 'PF']:
    t3 = df[(df['tier'] == 'Tier 3 - Development') & 
            (df['Position'] == pos)].copy()
    
    t3['height_inches'] = t3['Height'].apply(height_to_inches)
    
    year_score = {
        'FR':    5, 'RS-FR': 5,
        'SO':    4, 'RS-SO': 4,
        'JR':    3, 'RS-JR': 3,
        'SR':    1, 'RS-SR': 1,
    }
    t3['year_score'] = t3['Year'].map(year_score).fillna(3)
    
    # Youth 60%, Height 40%
    t3['dev_score'] = (
        (t3['year_score'] * 10 * 0.60) +
        (t3['height_inches'] * 0.40)
    )
    t3 = t3.sort_values('dev_score', ascending=False)
    
    print(f"\n  {pos} DEVELOPMENT PROSPECTS")
    print(f"  {'Name':<25} {'Year':<8} {'Height':<8} {'Weight':<8} {'Hometown'}")
    print(f"  {'-'*70}")
    
    for _, row in t3.iterrows():
        hometown = str(row['Hometown']) if pd.notna(row['Hometown']) else 'N/A'
        weight = str(int(row['Weight'])) if pd.notna(row['Weight']) else 'N/A'
        print(f"  {row['Name']:<25} {row['Year']:<8} {row['Height']:<8} {weight:<8} {hometown}")


  DEVELOPMENT PROSPECTS — C AND PF
  Ranked by: Youth first, then Height

  C DEVELOPMENT PROSPECTS
  Name                      Year     Height   Weight   Hometown
  ----------------------------------------------------------------------
  Nico Onyekwere            FR       7-1      240      Glen Head, NY
  Justin Houser             FR       6-11     N/A      Camp Hill, PA
  Paul Mbiya                FR       6-11     240      Kinshasa
  William Patterson         RS-SO    7-1      220      Brooklyn, NY
  Emmanuel Stephen          SO       7-0      215      Glendale, AZ
  Emeka Opurum              SO       7-0      200      Lagos
  Steven Solano             SO       6-11     230      Virginia Beach, VA
  Godswill Erheriene        SO       6-9      225      Glen Head, NY
  Stephen Osei              SO       6-9      200      Toronto, ON
  Xander Wedlow             SO       6-9      225      Detroit, MI
  Gai Chol                  JR       7-0      235      Decatur, GA
  Braden Pierce     

In [4]:
# HTML RECRUITING BOARD

html = """
<!DOCTYPE html>
<html>
<head>
<style>
   body { font-family: Arial, sans-serif; margin: 40px; color: #333; }
   .header { background-color: #CC0000; color: white; padding: 20px; text-align: center; }
   .header h1 { margin: 0; font-size: 28px; }
   .header h2 { margin: 5px 0 0 0; font-size: 16px; font-weight: normal; }
   .author { text-align: right; font-size: 12px; margin-top: 10px; color: #666; }
   .position-header { background-color: #333; color: white; padding: 10px; 
                      margin-top: 30px; font-size: 18px; }
   .tier-label { color: #CC0000; font-weight: bold; margin: 15px 0 5px 0; }
   table { width: 100%; border-collapse: collapse; margin-bottom: 10px; }
   th { background-color: #f0f0f0; padding: 8px; text-align: left; 
        border-bottom: 2px solid #CC0000; font-size: 13px; }
   td { padding: 7px 8px; border-bottom: 1px solid #ddd; font-size: 13px; }
   tr:hover { background-color: #fff9f9; }
   .tier2-label { color: #666; font-weight: bold; margin: 15px 0 5px 0; }
   .footer { text-align: center; margin-top: 40px; font-size: 11px; color: #999; }
</style>
</head>
<body>
<div class="header">
   <h1>NC STATE WOLFPACK</h1>
   <h2>2026-27 Transfer Portal Recruiting Board</h2>
</div>
<p class="author">Prepared by: JJ Eubank Basketball Analyst | """ + pd.Timestamp.now().strftime('%B %d, %Y') + """</p>
"""

positions = ['C', 'SF', 'PF', 'PG', 'SG']
position_names = {
   'C': 'CENTER', 'SF': 'SMALL FORWARD',
   'PF': 'POWER FORWARD', 'PG': 'POINT GUARD', 'SG': 'SHOOTING GUARD'
}

for pos in positions:
   t1 = df[(df['tier'] == 'Tier 1') & (df['Position'] == pos)].copy()
   t1 = t1.dropna(subset=['fit_score'])
   t1 = t1.sort_values('fit_score', ascending=False).head(10)
   
   t2 = df[(df['tier'] == 'Tier 2') & (df['Position'] == pos)].copy()
   t2 = t2.sort_values('composite_score', ascending=False)

   html += f"""
<div class="position-header">{position_names[pos]}</div>
<div class="tier-label">TIER 1 - Full Advanced Metrics</div>
<table>
<tr>
   <th>#</th><th>Name</th><th>Previous School</th>
   <th>Fit Score</th><th>Defense</th><th>Offense</th><th>Year</th>
</tr>"""
   
   for i, (_, row) in enumerate(t1.iterrows(), 1):
       html += f"""
<tr>
   <td>{i}</td>
   <td><strong>{row['Name']}</strong></td>
   <td>{row['Team']}</td>
   <td>{row['fit_score']:.1f}</td>
   <td>{row['defense_score']:.1f}</td>
   <td>{row['offense_score']:.1f}</td>
   <td>{row['Year']}</td>
</tr>"""
   
   html += "</table>"
   
   if len(t2) > 0:
       html += """<div class="tier2-label">TIER 2 - ON3 Rating Only</div>
<table>
<tr><th>Name</th><th>Year</th><th>ON3 Rating</th></tr>"""
       for _, row in t2.iterrows():
           html += f"""
<tr>
   <td><strong>{row['Name']}</strong></td>
   <td>{row['Year']}</td>
   <td>{row['ON3 Rating']:.0f}</td>
</tr>"""
       html += "</table>"

html += """<div class="position-header">DEVELOPMENT PROSPECTS - C and PF</div>
<div class="tier-label">Physical Profile Only - Project Players</div>"""

def height_to_inches(h):
   try:
       parts = str(h).split('-')
       return int(parts[0]) * 12 + int(parts[1])
   except:
       return 0

year_score = {
   'FR': 5, 'RS-FR': 5, 'SO': 4, 'RS-SO': 4,
   'JR': 3, 'RS-JR': 3, 'SR': 1, 'RS-SR': 1
}

for pos in ['C', 'PF']:
   t3 = df[(df['tier'] == 'Tier 3 - Development') & (df['Position'] == pos)].copy()
   t3['height_inches'] = t3['Height'].apply(height_to_inches)
   t3['year_score'] = t3['Year'].map(year_score).fillna(3)
   t3['dev_score'] = (t3['year_score'] * 10 * 0.60) + (t3['height_inches'] * 0.40)
   t3 = t3.sort_values('dev_score', ascending=False)
   
   html += f"""
<div class="tier2-label">{pos}</div>
<table>
<tr><th>Name</th><th>Year</th><th>Height</th><th>Weight</th><th>Hometown</th></tr>"""
   
   for _, row in t3.iterrows():
       weight = str(int(row['Weight'])) if pd.notna(row['Weight']) else 'N/A'
       hometown = str(row['Hometown']) if pd.notna(row['Hometown']) else 'N/A'
       html += f"""
<tr>
   <td><strong>{row['Name']}</strong></td>
   <td>{row['Year']}</td>
   <td>{row['Height']}</td>
   <td>{weight}</td>
   <td>{hometown}</td>
</tr>"""
   html += "</table>"

html += """
<div class="footer">
   NC State Wolfpack Basketball | JJ Eubank Basketball Analytics | Confidential
</div>
</body>
</html>"""

with open("data/outputs/NC_State_Recruiting_Board.html", "w") as f:
   f.write(html)

print("File: data/outputs/NC_State_Recruiting_Board.html")


File: data/outputs/NC_State_Recruiting_Board.html
